<a href="https://colab.research.google.com/github/Abhhiiissshhek/100_days_of_ML_challenge/blob/main/day-07-logistic-regression/day07b_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Day 07 (B)**

## **Loading Data.....**

In [28]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# showing the total no. of missing values in the dataset
df.isnull().sum()

# showing the percentage of missing data
(df.isnull().sum() / len(df)) * 100

# Cabin is 77% missing - basically useless
df.drop('Cabin', axis=1, inplace=True)

# Age is 20% missing - fill with MEDIAN
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked is 0.2% missing - fill with MODE
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

df.isnull().sum().sum() # our dataset is clean now !!

# Create Family Size from SibSp + Parch
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Create Age Groups
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Middle-Aged', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

# Create IsAlone flag
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Embarked  FamilySize Title     AgeGroup  \
0      0         A/5 21171   7.2500        S           2    Mr        Adult   
1      0          PC 17599  71.2833        C           2   Mrs  Middle-Aged   
2      0  STON/O2. 3101282   7.9250        S           1  Miss        Adult   
3   

<>:28: SyntaxWarning: invalid escape sequence '\.'
<>:28: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-925696125.py:28: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
/tmp/ipython-input-925696125.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipython-input-925696125.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using

**This above code give us the featured csv....**

## **DATA LOADED ✅**

# **STEP 1: SIGMOID FUNCTION**

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Test
print(sigmoid(0))   # 0.5
print(sigmoid(5))   # 0.99
print(sigmoid(-5))  # 0.01
print(sigmoid(10000000))


0.5
0.9933071490757153
0.0066928509242848554
1.0


## **SIGMOID = SQUISH MACHINE 📈**

# **STEP 2: PREDICT FUNCTION**

In [29]:
def predict(X, weights):
    z = np.dot(X, weights)
    return sigmoid(z)

## **LINEAR → SIGMOID = PROBABILITY**

# **STEP 3: LOSS FUNCTION**

In [30]:
def loss(y_true, y_pred):
    # Clip to avoid log(0)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )

## **LOSS = HOW WRONG WE ARE 📉**

# **STEP 4: GRADIENT DESCENT**

In [31]:
def gradient_descent(X, y, weights, lr=0.01):
    m = len(X)
    y_pred = predict(X, weights)
    gradient = np.dot(X.T, (y_pred - y)) / m
    weights = weights - lr * gradient
    return weights

## **GRADIENT = LEARNING ⬇️**

# **STEP 5: TRAINING LOOP**

In [32]:
def train(X, y, lr=0.01, epochs=1000):
    X = np.c_[np.ones(X.shape[0]), X]  # Add bias
    weights = np.random.randn(X.shape[1])

    for i in range(epochs):
        weights = gradient_descent(X, y, weights, lr)

        if i % 100 == 0:
            y_pred = predict(X, weights)
            print(f"Epoch {i}: Loss = {loss(y, y_pred):.4f}")

    return weights

## **1000 EPOCHS = 1000 UPDATES 🔄**

# **STEP 6: TRAIN + PREDICT**

In [18]:

# ✅ YEH LINES ADD KARO (X and y define karo)
features = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone']
X = df[features].copy()
y = df['Survived'].values

# Convert Sex to numbers
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
X = X.values

# Ab train/test split karo
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
# Train
weights = train(X_train, y_train)

# Predict on test
X_test_bias = np.c_[np.ones(X_test.shape[0]), X_test]
y_prob = predict(X_test_bias, weights)
y_pred = (y_prob >= 0.5).astype(int)

# Accuracy
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy:.2%}")

Epoch 0: Loss = 8.9951
Epoch 100: Loss = 1.3347
Epoch 200: Loss = 1.2619
Epoch 300: Loss = 1.6144
Epoch 400: Loss = 0.9010
Epoch 500: Loss = 1.1987
Epoch 600: Loss = 1.4109
Epoch 700: Loss = 0.8401
Epoch 800: Loss = 1.1594
Epoch 900: Loss = 1.2696
Accuracy: 75.98%


## **75.98% vs 79.89% 🔥**